In [6]:
from goatools.base import download_ncbi_associations
from goatools.obo_parser import GODag
from goatools.anno.genetogo_reader import Gene2GoReader

fin_gene2go = download_ncbi_associations()

  EXISTS: gene2go


In [7]:
obodag = GODag("gene2go")

gene2go: fmt(None) rel(None) 0 GO Terms


In [ ]:
# Read NCBI's gene2go. Store annotations in a list of namedtuples
objanno = Gene2GoReader(fin_gene2go, taxids=[10090])

# Get namespace2association where:
#    namespace is:
#        BP: biological_process               
#        MF: molecular_function
#        CC: cellular_component
#    assocation is a dict:
#        key: NCBI GeneID
#        value: A set of GO IDs associated with that gene
ns2assoc = objanno.get_ns2assc()

for nspc, id2gos in ns2assoc.items():
    print("{NS} {N:,} annotated mouse genes".format(NS=nspc, N=len(id2gos)))

In [1]:
import pandas as pd

In [2]:
df = pd.read_table('diffexp/venn-DEGs.csv',sep=',',index_col=[0,1,2])

In [3]:
idx = pd.IndexSlice

In [4]:
genes = df.loc[
    idx['all_neurons','(MDMAvsMeth_up&MDMAvsSaline_up)-MethvsSaline_down'], :
].index.to_list()

/var/folders/sx/tm64_b5n1vl137l9nnryg2zm0000gn/T/ipykernel_31861/133489078.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  genes = df.loc[


In [5]:
%reload_ext rpy2.ipython

In [6]:
%%R 
library(HPAanalyze)
library(tidyverse)

── Attaching packages ─────────────────────────────────────── tidyverse 1.3.2 ──
✔ ggplot2 3.3.6      ✔ purrr   0.3.5 
✔ tibble  3.1.8      ✔ dplyr   1.0.10
✔ tidyr   1.2.1      ✔ stringr 1.4.1 
✔ readr   2.1.3      ✔ forcats 0.5.2 
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()


In [7]:
%%R
downloadedData <- hpaDownload(downloadList='histology', version='example')
sapply(downloadedData, nrow)

R[write to console]: Only the followings are example/built-in datasets: 
 - Normal tissue 
 - Pathology 
 - Subcellular location 
Other datasets will not be loaded



       normal_tissue            pathology subcellular_location 
             1118517               393400                12813 


In [8]:
%%R
downloadedData$normal_tissue$tissue %>% unique

 [1] "adipose tissue"    "adrenal gland"     "appendix"         
 [4] "bone marrow"       "breast"            "bronchus"         
 [7] "caudate"           "cerebellum"        "cerebral cortex"  
[10] "cervix, uterine"   "colon"             "duodenum"         
[13] "endometrium 1"     "endometrium 2"     "epididymis"       
[16] "esophagus"         "fallopian tube"    "gallbladder"      
[19] "heart muscle"      "hippocampus"       "kidney"           
[22] "liver"             "lung"              "lymph node"       
[25] "nasopharynx"       "oral mucosa"       "ovary"            
[28] "pancreas"          "parathyroid gland" "placenta"         
[31] "prostate"          "rectum"            "salivary gland"   
[34] "seminal vesicle"   "skeletal muscle"   "skin 1"           
[37] "skin 2"            "small intestine"   "smooth muscle"    
[40] "soft tissue 1"     "soft tissue 2"     "spleen"           
[43] "stomach 1"         "stomach 2"         "testis"           
[46] "thyroid gland"     

In [9]:
%%R -i genes -o hpa_ann_subcellular_location,hpa_ann_normal_tissue
geneList <- genes
tissueList <- c(
    # 'adrenal gland',
    'cerebellum','cerebral cortex','hippocampus','hypothalamus','pituitary gland'
)

subsetData <- hpaSubset(data=downloadedData,
                         targetGene=geneList,
                         targetTissue=tissueList,
                         # targetCancer=cancerList,
                         # targetCellLine=cellLineList
                       )
hpa_ann_subcellular_location = subsetData$subcellular_location %>% data.frame
hpa_ann_normal_tissue = subsetData$normal_tissue
# subsetData

In [10]:
# %%R
# hpaVis(
#     downloadedData,
#     genes
# )

- https://pypi.org/project/mygene/#description
- https://www.kaggle.com/code/alexandervc/genes-info-with-mygene-python-package

In [40]:
import mygene

In [85]:
mg = mygene.MyGeneInfo()

In [92]:
search = mg.querymany(hpa_ann_normal_tissue.ensembl,as_dataframe=True)

querying 1-909...done.
Finished.
77 input query terms found dup hits:
	[('ENSG00000018236', 25), ('ENSG00000046653', 8), ('ENSG00000047849', 25), ('ENSG00000048649', 9), (
Pass "returnall=True" to return complete lists of duplicate or missing query terms.


In [100]:
hpa_ann_normal_tissue['name'] = search.name.to_list()

In [102]:
out = hpa_ann_normal_tissue[
    ['gene','name','tissue','cell_type','level']
].set_index(['gene','name','tissue','cell_type'])

out[out.level != 'Not detected'].to_excel('annotated/sample-HPA.xlsx')

In [79]:
# pd.DataFrame(hpa_ann_normal_tissue.groupby('gene'))
# # set(hpa_ann_subcellular_location.main_location)
# set(hpa_ann_normal_tissue.cell_type)

In [103]:
!date

Thu Nov 10 00:51:13 PST 2022
